In [1]:
import os
import sys
import time
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
import warnings
warnings.filterwarnings('ignore')

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_recall_curve, classification_report
from sklearn.datasets import make_classification
from xgboost import XGBClassifier
from xgboost import plot_importance
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

import lightgbm as lgbm

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from utils import user_utils

In [ ]:
raw_df = pd.read_csv('../data/creditcard.csv')

In [2]:
def cap_outliers(df, columns=None, weight=1.5):
    """
    IQR 기반으로 이상치를 탐지하고 상한/하한 값으로 대체(Capping)하는 함수.
    
    df: DataFrame
    columns: 특정 컬럼 리스트 (None이면 전체 수치형 컬럼)
    weight: IQR 배수 (기본 1.5)
    """
    df_capped = df.copy()
    
    # 처리할 컬럼 선택
    if columns is None:
        columns = df_capped.select_dtypes(include=np.number).columns
    
    for col in columns:
        # 사분위수 및 IQR 계산
        Q1 = df_capped[col].quantile(0.25)
        Q3 = df_capped[col].quantile(0.75)
        IQR = Q3 - Q1
        
        # 이상치 경계 계산
        lower_bound = Q1 - weight * IQR
        upper_bound = Q3 + weight * IQR
        
        # Capping 적용
        df_capped[col] = df_capped[col].clip(lower=lower_bound, upper=upper_bound)
        
    return df_capped

In [ ]:
amount_n = np.log1p(raw_df['Amount'])
raw_df.insert(0,'Amount Scaled', amount_n)
raw_df.drop(['Amount'], axis=1, inplace=True)

In [ ]:
raw_df.to_csv('../data/scaledAmount_data.csv',index=False)

In [3]:
scaled_df = pd.read_csv('../data/scaledAmount_data.csv')

temp_X_feature = scaled_df.iloc[:,:-1]
y_label = scaled_df.iloc[:,-1]

X_feature = cap_outliers(temp_X_feature)

In [4]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_feature, y_label)

print("\n--- SMOTE 오버샘플링 후 ---")
print(y_resampled.value_counts())

Exception in thread Thread-6 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\tj\.conda\envs\ml_dev\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "c:\Users\tj\.conda\envs\ml_dev\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\tj\.conda\envs\ml_dev\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc0 in position 4: invalid start byte



--- SMOTE 오버샘플링 후 ---
Class
0    284315
1    284315
Name: count, dtype: int64


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled,
    y_resampled,
    test_size = 0.2,
    random_state = 42
)

In [5]:
from sklearn.svm import SVC

# 5. Linear SVM
svc_liner_basic_params = {
  'C' : 1.0, 
  'kernel' : "linear", 
  'class_weight' : "balanced"
}
linear_svm = SVC(**svc_liner_basic_params)  # 불균형 데이터라 balanced 권장

In [ ]:
user_utils.get_model_train_eval(linear_svm, 'SVM_smote_capped_ejm', X_train, X_test, y_train, y_test)